# core

> Async Google Workspace clients built from Google's Discovery API.

In [ ]:
#| default_exp core

In [ ]:
#| export
from collections.abc import Sized
from fastcore.utils import *
from fastgws.auth import *
from fastspec.spec import SpecParser
from fastspec.oapi import AsyncTransport, OpFunc, _build_groups

import httpx, os

In [ ]:
#| hide
from nbdev.showdoc import *

In [ ]:
#| export
class GWSObject(AttrDict):
    def __repr__(self):
        keys = 'id name title summary emailAddress threadId documentId spreadsheetId sheetId mimeType kind'.split()
        bits = [f"{k}={self[k]!r}" for k in keys if k in self]
        bits += [f"{k}={len(v)}" for k,v in self.items()
                 if isinstance(v, Sized) and not isinstance(v, (str, bytes)) and k not in keys]
        return f"{type(self).__name__}({', '.join(bits)})"

def gclasses(doc):
    res = {}
    for n,s in doc.get('schemas', {}).items():
        kind = nested_idx(s, 'properties', 'kind', 'default')
        if kind: res[kind] = type(n, (GWSObject,), {})
    return res

def g2obj(x, gcls):
    if isinstance(x, list): return L(x).map(lambda o: g2obj(o, gcls))
    if not isinstance(x, dict): return x
    cls = gcls.get(x.get('kind'), GWSObject)
    return cls({k:g2obj(v, gcls) for k,v in x.items()})

In [ ]:
#| export
class GWSTransport(AsyncTransport):
    def __init__(self, *args, gcls=None, **kwargs):
        super().__init__(*args, **kwargs)
        self.gcls = gcls or {}

    async def request(self, *args, raw=False, **kwargs):
        res = await super().request(*args, raw=raw, **kwargs)
        return res if raw else g2obj(res, self.gcls)

class GWSApi:
    service = None

    def __init__(self, service=None, version=None, token=None, creds=None,
                 api_key=None, headers=None, timeout=60.0, doc=None):
        service = ifnone(service, self.service)
        if service is None: raise ValueError('`service` is required')
        self.service,self.version = service,version
        self.doc = doc or self._get_doc(service, version)
        self.spec = SpecParser.from_discovery(self.doc)
        self.gcls = gclasses(self.doc)

        hdrs = headers or {}
        if creds or token: hdrs = {**auth_headers(creds, token), **hdrs}
        if api_key is None and not (creds or token): api_key = os.getenv('GOOGLE_API_KEY', os.getenv('GWS_API_KEY'))
        if api_key: hdrs = {'X-Goog-Api-Key': api_key, **hdrs}

        self.transport = GWSTransport(timeout=timeout, base_headers=hdrs, gcls=self.gcls)
        self.ops = [OpFunc(o, self.transport, self.spec.base_url, noop) for o in self.spec.ops]
        self.func_dict = {f"{o.path}:{o.verb.upper()}": o for o in self.ops}
        self.groups = _build_groups(self.ops)
        for k,v in self.groups.items(): setattr(self, k, v)

    def _get_doc(self, service, version=None):
        if version:
            url = f'https://www.googleapis.com/discovery/v1/apis/{service}/{version}/rest'
        else:
            apis = httpx.get('https://discovery.googleapis.com/discovery/v1/apis').json()['items']
            api = first(a for a in apis if a['name'] == service and a.get('preferred'))
            url = api['discoveryRestUrl']
            self.version = api['version']
        return httpx.get(url).json()

In [ ]:
creds = oauth_creds(scopes=['https://www.googleapis.com/auth/gmail.readonly',
                            'https://www.googleapis.com/auth/drive.readonly'],
                    redirect_uri='https://oauth.appapis.org/redirect')

In [ ]:
drive = GWSApi('drive', creds=creds)
fs = await drive.files.list(q="name contains 'fastgws' and trashed=false", page_size=10)
fs, fs.files[0]

(FileList(kind='drive#fileList', files=2),
 File(id='1ohlMW4OttiYNExpTR4SfVdDpGwW7qvyJAYY-YcxaYW0', name='fastgws test doc', mimeType='application/vnd.google-apps.document', kind='drive#file'))

In [ ]:
gmail = GWSApi('gmail', creds=creds)
msgs = await gmail.users.messages.list(user_id='me', max_results=10)
msgs

<div class="prose" markdown="1">

```python
GWSObject(messages=10)
```

</div>

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()